# 00 · Data checks
Run this **once** before anything else. It checks that every ERA5 file is present, that all pieces line up on
one grid, and that IBTrACS loads correctly.

**Runtime:** CPU (standard is fine). **Takes:** a few minutes.
**Output:** `outputs/checks/missing_months.csv`

In [ ]:
# ▶ Setup: connect Google Drive, install missing packages, load the project code
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost", "lightgbm", "xgboost"], check=False)

PROJECT_DIR = "/content/drive/MyDrive/CycloneProject"      # <- change only if you put the project elsewhere
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import numpy as np, pandas as pd
from src import config as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)
print("project:", C.PROJECT, "| ERA5:", C.ERA5_ROOT)

## 1. Missing ERA5 months
Every variable, every buffer strip and geopotential, for all years in `C.YEARS` (1990–2023) **plus** the extra
recent test months in `C.RECENT_TEST_MONTHS` (Oct–Dec 2025: Montha, Ditwah).

In [ ]:
from src import checks
miss = checks.missing_months()
miss.to_csv(C.out("checks") / "missing_months.csv", index=False)
if miss.empty:
    print("ALL CLEAR - no missing files for", C.YEARS, "+ recent test months", C.RECENT_TEST_MONTHS)
else:
    print(f"{len(miss)} missing files. Summary:")
    display(checks.summarise_missing(miss))
    print("First missing months:"); display(miss.head(20))

## 2. Grid check for one month
Merges main file + 4 strips for each variable. `filled_fraction` should be **1.0** (SST is lower because it is empty over land). Check the pressure levels listed.

In [ ]:
grid = checks.grid_check(1990, 1)
display(grid)

## 3. IBTrACS
Download `ibtracs.NI.list.v04r01.csv` from NOAA NCEI and put it in `CycloneProject/data/`.
**Do not open and save it in Excel** – Excel rewrites the dates day-first (the loader copes, but a fresh copy is safest).
Recent storms (2024 onward) are marked `PROVISIONAL` / `US-PROVISIONAL` in `TRACK_TYPE`; the loader keeps them.

In [ ]:
from src import tracks
fixes = tracks.load_ibtracs()
per_year = checks.ibtracs_summary(fixes)
display(per_year.to_frame().T)

## 4. Splits and test storms
Check the split years and that the recent test storms (Montha, Ditwah) are found.

In [ ]:
print({k: C.span(getattr(C, k)) for k in ("TRAIN", "VAL", "CAL", "TEST")}, "| recent test months:", C.RECENT_TEST_MONTHS)
recent = set(map(tuple, C.RECENT_TEST_MONTHS))
f = fixes.assign(_recent=[(t.year, t.month) in recent for t in fixes.ISO_TIME])
storms = f.groupby("SID").agg(name=("NAME", "first"), start=("ISO_TIME", "min"), end=("ISO_TIME", "max"),
                              fixes=("ISO_TIME", "size"), peak_kt=("wind", "max"), recent=("_recent", "any"))
storms["year"] = storms.start.dt.year
main = storms[storms.year.between(*C.TEST)].sort_values("start")
rec = storms[storms.recent].sort_values("start")
print(f"\nmain test storms {C.span(C.TEST)}: {len(main)} ({int((main.peak_kt >= 34).sum())} reached cyclonic-storm strength)")
display(main)
print(f"recent test storms: {len(rec)}")
display(rec)
for want in ("MONTHA", "DITWAH"):
    print(want, "found" if (rec.name.str.upper() == want).any() else "NOT FOUND - check IBTrACS file / RECENT_TEST_MONTHS")

**Next:** when section 1 says *ALL CLEAR*, open `01_build_occurrence_table.ipynb`.